### Load IOCs

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load brute force IOCs
with open('iocs_brute_force.json', 'r') as f:
    iocs = json.load(f)

# Display metadata
print("="*60)
print("BRUTE FORCE ATTACK ANALYSIS")
print("="*60)
for key, value in iocs['metadata'].items():
    print(f"{key:20s}: {value}")

###  Attack Summary

In [ ]:
# Create DataFrame
attacks_df = pd.DataFrame(iocs['brute_force_attacks'])

print("\n" + "="*60)
print("ATTACK SUMMARY")
print("="*60)
print(attacks_df[['attacker_ip', 'failed_attempts', 'breach_occurred', 'severity']])

### Failed Attempts Visualization

In [ ]:
# Bar chart of failed attempts
plt.figure(figsize=(12, 6))

for attack in iocs['brute_force_attacks']:
    usernames = attack['usernames_tried']
    # Simulate distribution (in real case, parse from logs)
    counts = [attack['failed_attempts'] // len(usernames)] * len(usernames)
    
    plt.bar(usernames, counts, color='#E74C3C', edgecolor='black', linewidth=1.5)

plt.title('Failed Login Attempts by Username', fontsize=16, fontweight='bold')
plt.xlabel('Username', fontsize=12)
plt.ylabel('Failed Attempts', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('failed_logins_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("[+] Chart saved: failed_logins_chart.png")

### Attack Timeline

In [ ]:
from datetime import datetime

for attack in iocs['brute_force_attacks']:
    first = datetime.fromisoformat(attack['first_attempt'])
    last = datetime.fromisoformat(attack['last_attempt'])
    duration = attack['attack_duration_seconds']
    
    print(f"\n{'='*60}")
    print(f"ATTACK TIMELINE: {attack['attacker_ip']}")
    print(f"{'='*60}")
    print(f"First Attempt:     {first.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Last Attempt:      {last.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total Duration:    {duration:.2f} seconds ({duration/60:.2f} minutes)")
    print(f"Attack Rate:       {attack['failed_attempts'] / (duration/60):.2f} attempts/min")
    
    if attack['breach_occurred']:
        print(f"\n BREACH STATUS:   CONFIRMED")
        print(f"Compromised User:  {attack['compromised_account']}")
        print(f"MITRE ATT&CK:      {', '.join(attack['mitre_attack'])}")
    else:
        print(f"\n BREACH STATUS:   PREVENTED")

### Attack Chain Visualization

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))

# Timeline boxes
boxes = [
    {'x': 0, 'label': 'Port Scan\n(T1046)', 'color': '#3498DB'},
    {'x': 1, 'label': '8 min\ndelay', 'color': '#95A5A6'},
    {'x': 2, 'label': 'Brute Force\n(T1110.001)', 'color': '#E67E22'},
    {'x': 3, 'label': 'Breach\n(T1078)', 'color': '#E74C3C'},
]

for box in boxes:
    rect = mpatches.FancyBboxPatch((box['x'], 0.3), 0.8, 0.4,
                                   boxstyle="round,pad=0.1",
                                   edgecolor='black',
                                   facecolor=box['color'],
                                   linewidth=2)
    ax.add_patch(rect)
    ax.text(box['x'] + 0.4, 0.5, box['label'],
           ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    
    # Arrows
    if box['x'] < 3:
        ax.arrow(box['x'] + 0.85, 0.5, 0.1, 0,
                head_width=0.1, head_length=0.05, fc='black', ec='black')

ax.set_xlim(-0.2, 4)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Attack Kill Chain: Reconnaissance → Exploitation', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('attack_chain_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("[+] Chart saved: attack_chain_visualization.png")